# Binary ViT-Small (ViT-S/16) Dermoscope-Only — 5-Fold Cross-Validation (Full Fine-Tune)

K-Fold CV version of the full fine-tuning experiment using ViT-Small. Instead of a single train/val split, we train 5 independent models and average their results for more robust performance estimates (mean ± std).

**Protocol:** Hold out 10% test set → 5-fold CV on remaining 90% → each fold: warmup (frozen, 5 epochs) + full fine-tune entire backbone (up to 30 epochs) → evaluate all folds on shared test set → ensemble by averaging probabilities.

**Binary labels:**
| Original Class | Binary Label |
|---|---|
| Melanoma | **Malignant** (1) |
| BCC | **Malignant** (1) |
| SCC | **Malignant** (1) |
| Actinic Keratosis | **Malignant** (1) |
| Malignant_Other | **Malignant** (1) |
| Melanocytic_Nevus | **Benign** (0) |
| Seborrheic_Keratosis | **Benign** (0) |
| Dermatofibroma | **Benign** (0) |
| Hemangioma | **Benign** (0) |
| Fibrous_Papule | **Benign** (0) |
| Other_Benign | **Benign** (0) |

## 1. Setup & Imports

In [2]:
import os
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, precision_recall_fscore_support, fbeta_score
)

from transformers import ViTForImageClassification

from utils import (
    BinaryDermoscopeSkinDataset,
    get_transforms,
    BINARY_MAP,
    CLASS_NAMES,
    BinaryFocalLossWithSmoothing,
    mixup_data,
    mixup_criterion,
    find_optimal_threshold,
    compute_metrics,
    MIN_SENSITIVITY_TARGET,
)

print("Imports complete.")

Imports complete.


In [3]:
# ── Configuration ──
IMAGES_DIR = "../DataCleaning/Images"
MANIFEST_PATH = "../DataCleaning/instances.csv"
BATCH_SIZE = 12
SEED = 42
K_FOLDS = 5

TRAIN_THRESHOLD = 0.5

# Training config
WARMUP_EPOCHS = 5
FT_EPOCHS = 30
PATIENCE = 8
MIXUP_ALPHA = 0.15

LR_BACKBONE = 2e-5
LR_HEAD = 1e-4
LABEL_SMOOTHING = 0.05

NUM_WORKERS = 0 if os.name == "nt" else min(os.cpu_count() or 8, 12)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print(f"Using: {DEVICE}")
print(f"Mixed precision: {USE_AMP}")
print(f"PyTorch: {torch.__version__}")
print(f"\n🧪 STRATIFIED {K_FOLDS}-FOLD CV — FULL BACKBONE FINE-TUNE")
print(f"  • Phase 2 unfreezes ENTIRE backbone")
print(f"  • {K_FOLDS} folds, each with warmup + fine-tuning")
print(f"  • Reports mean ± std of all metrics across folds")

if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using: cuda
Mixed precision: True
PyTorch: 2.5.1+cu121

🧪 STRATIFIED 5-FOLD CV — FULL BACKBONE FINE-TUNE
  • Phase 2 unfreezes ENTIRE backbone
  • 5 folds, each with warmup + fine-tuning
  • Reports mean ± std of all metrics across folds

GPU: NVIDIA GeForce RTX 4060
GPU Memory: 8.6 GB


## 2. Transforms

In [4]:
train_transform, eval_transform = get_transforms(image_size=224, resize_size=256)

print("✓ Transforms created using utils.get_transforms()")

✓ Transforms created using utils.get_transforms()


## 3. Load Data & Hold Out Test Set

In [5]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Total instances: {len(manifest)}")
print(f"\nOriginal class distribution:")
print(manifest["cancer_type"].value_counts())

manifest["binary_label"] = manifest["cancer_type"].map(BINARY_MAP)
print(f"\nBinary distribution:")
for lbl, name in enumerate(CLASS_NAMES):
    count = (manifest["binary_label"] == lbl).sum()
    pct = 100 * count / len(manifest)
    print(f"  {name} ({lbl}): {count} ({pct:.1f}%)")

# Hold out 10% test set (IDENTICAL to baseline)
sss_trainval_test = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
trainval_idx, test_idx = next(sss_trainval_test.split(manifest, manifest["binary_label"]))

trainval_manifest = manifest.iloc[trainval_idx].reset_index(drop=True)
test_manifest = manifest.iloc[test_idx].reset_index(drop=True)

print(f"\nHeld-out test set: {len(test_manifest)} instances")
print(f"Remaining for K-Fold CV: {len(trainval_manifest)} instances")

test_dataset = BinaryDermoscopeSkinDataset(
    test_manifest, IMAGES_DIR, transform=eval_transform
)
loader_kwargs = {'num_workers': NUM_WORKERS, 'pin_memory': True} if torch.cuda.is_available() else {}
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
print(f"Test dataset samples: {len(test_dataset)}")

Total instances: 906

Original class distribution:
cancer_type
BCC                     200
Melanocytic_Nevus       190
Other_Benign            138
SCC                     123
Seborrheic_Keratosis     79
Melanoma                 77
Actinic_Keratosis        63
Dermatofibroma           16
Hemangioma               10
Fibrous_Papule            6
Malignant_Other           4
Name: count, dtype: int64

Binary distribution:
  Benign (0): 439 (48.5%)
  Malignant (1): 467 (51.5%)

Held-out test set: 91 instances
Remaining for K-Fold CV: 815 instances
Test dataset samples: 91


## 4. Model Architecture

In [6]:
class BinaryViTModel(nn.Module):
    """
    ViT-Small backbone with custom classification head.
    Uses HuggingFace transformers ViTForImageClassification as base.
    
    ViT-Small has embed_dim=384 (vs ViT-Base's 768).
    """
    def __init__(self, vit_model, embed_dim=384, hidden=512, dropout=0.3):
        super().__init__()
        self.vit = vit_model.vit  # Extract the ViT backbone (without classification head)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, x):
        # ViT expects pixel_values
        outputs = self.vit(pixel_values=x)
        # Use CLS token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logit = self.classifier(cls_embedding)
        return logit.squeeze(-1)


print("✓ BinaryViTModel class defined.")

✓ BinaryViTModel class defined.


## 5. Load Pretrained ViT-Small Backbone (Reference Copy)

In [7]:
# Load once — we'll deep copy this for each fold so every fold starts from pretrained weights
vit_model_ref = ViTForImageClassification.from_pretrained(
    "WinKawaks/vit-small-patch16-224",
    num_labels=1,
    ignore_mismatched_sizes=True
)
embed_dim = vit_model_ref.config.hidden_size  # 384 for ViT-Small
print(f"ViT-Small (patch16-224) loaded. Embed dim: {embed_dim}")
print("This model will be deep-copied for each fold.")

config.json: 0.00B [00:00, ?B/s]

c:\Coding\SkinCancerImageClassificationResearch\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Reyna\.cache\huggingface\hub\models--WinKawaks--vit-small-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You passed `num_labels=1` which is incompatible to the `id2label` map of length `10

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 384]) vs model:torch.Size([1, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([1])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT-Small (patch16-224) loaded. Embed dim: 384
This model will be deep-copied for each fold.


## 6. Stratified 5-Fold Cross-Validation Training Loop

Each fold:
1. **Phase 1** — Warmup (frozen backbone, train head only, 5 epochs)
2. **Phase 2** — Full fine-tune entire backbone (up to 30 epochs, early stopping on val AUC-ROC)
3. **Evaluate** on both the fold's validation set and the shared held-out test set

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Storage for per-fold results
fold_val_results = []   # validation fold metrics
fold_test_results = []  # test set metrics (each fold's best model evaluated on held-out test)
fold_histories = []     # training curves per fold
fold_thresholds = []    # optimal thresholds per fold
fold_test_probs = []    # test set probabilities per fold (for ensemble later)

print(f"\n{'='*70}")
print(f"  STRATIFIED {K_FOLDS}-FOLD CROSS-VALIDATION")
print(f"{'='*70}\n")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(trainval_manifest, trainval_manifest["binary_label"])):
    print(f"\n{'#'*70}")
    print(f"  FOLD {fold_idx + 1} / {K_FOLDS}")
    print(f"{'#'*70}")

    # ── Seed reset for reproducibility within each fold ──
    torch.manual_seed(SEED + fold_idx)
    np.random.seed(SEED + fold_idx)
    random.seed(SEED + fold_idx)

    # ── Split ──
    fold_train_manifest = trainval_manifest.iloc[train_idx].reset_index(drop=True)
    fold_val_manifest = trainval_manifest.iloc[val_idx].reset_index(drop=True)
    print(f"  Train: {len(fold_train_manifest)} | Val: {len(fold_val_manifest)}")

    for lbl, name in enumerate(CLASS_NAMES):
        tr_cnt = (fold_train_manifest["binary_label"] == lbl).sum()
        va_cnt = (fold_val_manifest["binary_label"] == lbl).sum()
        print(f"    {name}: train={tr_cnt}, val={va_cnt}")

    # ── Datasets & loaders ──
    fold_train_dataset = BinaryDermoscopeSkinDataset(
        fold_train_manifest, IMAGES_DIR, transform=train_transform
    )
    fold_val_dataset = BinaryDermoscopeSkinDataset(
        fold_val_manifest, IMAGES_DIR, transform=eval_transform
    )

    train_labels = [label for _, label in fold_train_dataset]
    class_counts = {0: train_labels.count(0), 1: train_labels.count(1)}
    total_train = len(train_labels)
    class_weights = {cls: total_train / count for cls, count in class_counts.items()}
    sample_weights = torch.tensor([class_weights[label] for label in train_labels], dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler, **loader_kwargs)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

    print(f"  Dataset samples — Train: {len(fold_train_dataset)} | Val: {len(fold_val_dataset)}")

    # ── Fresh model from pretrained backbone ──
    vit_copy = copy.deepcopy(vit_model_ref)
    model = BinaryViTModel(
        vit_model=vit_copy, embed_dim=embed_dim, hidden=512, dropout=0.3
    ).to(DEVICE)

    # ── Pos weight for loss ──
    pos_weight = torch.tensor([class_counts[0] / class_counts[1]]).to(DEVICE)
    criterion = BinaryFocalLossWithSmoothing(gamma=2.0, pos_weight=pos_weight, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 1: Warmup (Frozen Backbone)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  ── Phase 1: Warmup ({WARMUP_EPOCHS} epochs, frozen backbone) ──")
    for param in model.vit.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer_warmup = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler_warmup = CosineAnnealingLR(optimizer_warmup, T_max=WARMUP_EPOCHS)

    for epoch in range(WARMUP_EPOCHS):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in fold_train_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                optimizer_warmup.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer_warmup)
                scaler.update()
            else:
                logits = model(imgs)
                loss = criterion(logits, labels)
                optimizer_warmup.zero_grad()
                loss.backward()
                optimizer_warmup.step()
            train_loss += loss.item()
            preds = (torch.sigmoid(logits) >= TRAIN_THRESHOLD).long()
            total += labels.size(0)
            correct += preds.eq(labels.long()).sum().item()
        scheduler_warmup.step()
        avg_loss = train_loss / len(fold_train_loader)
        acc = 100. * correct / total
        print(f"    Warmup Epoch [{epoch+1}/{WARMUP_EPOCHS}] Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 2: Fine-Tune Entire Backbone (Early Stopping on AUC-ROC)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n  ── Phase 2: Fine-tuning entire backbone ({FT_EPOCHS} max epochs) ──")
    # Unfreeze entire backbone
    for param in model.vit.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True

    optimizer_ft = torch.optim.AdamW([
        {'params': model.vit.parameters(), 'lr': LR_BACKBONE},
        {'params': model.classifier.parameters(), 'lr': LR_HEAD},
    ], weight_decay=1e-4)
    scheduler_ft = CosineAnnealingLR(optimizer_ft, T_max=FT_EPOCHS)

    best_val_auc = 0.0
    best_model_state = None
    epochs_no_improve = 0
    ft_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
                  'val_auc': [], 'val_sens': [], 'val_spec': []}

    for epoch in range(FT_EPOCHS):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for imgs, labels in fold_train_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if MIXUP_ALPHA > 0 and np.random.rand() < 0.5:
                imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)
                use_mixup = True
            else:
                use_mixup = False
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    if use_mixup:
                        loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                    else:
                        loss = criterion(logits, labels)
                optimizer_ft.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer_ft)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer_ft)
                scaler.update()
            else:
                logits = model(imgs)
                if use_mixup:
                    loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                else:
                    loss = criterion(logits, labels)
                optimizer_ft.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer_ft.step()
            train_loss += loss.item()
            preds = (torch.sigmoid(logits) >= TRAIN_THRESHOLD).long()
            if use_mixup:
                total += labels_a.size(0)
                correct += (lam * preds.eq(labels_a.long()).float() +
                           (1 - lam) * preds.eq(labels_b.long()).float()).sum().item()
            else:
                total += labels.size(0)
                correct += preds.eq(labels.long()).sum().item()
        scheduler_ft.step()
        avg_train_loss = train_loss / len(fold_train_loader)
        train_acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        val_preds, val_true, val_probs = [], [], []
        with torch.no_grad():
            for imgs, labels in fold_val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True).float()
                if USE_AMP:
                    with torch.amp.autocast('cuda'):
                        logits = model(imgs)
                        loss = criterion(logits, labels)
                else:
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                val_loss += loss.item()
                probs = torch.sigmoid(logits)
                preds = (probs >= TRAIN_THRESHOLD).long()
                total += labels.size(0)
                correct += preds.eq(labels.long()).sum().item()
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.long().cpu().numpy())
                val_probs.extend(probs.cpu().numpy())
        avg_val_loss = val_loss / len(fold_val_loader)
        val_acc = 100. * correct / total
        val_auc = roc_auc_score(val_true, val_probs)
        tn, fp, fn, tp = confusion_matrix(val_true, val_preds).ravel()
        val_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        val_spec = tn / (tn + fp) if (tn + fp) > 0 else 0

        ft_history['train_loss'].append(avg_train_loss)
        ft_history['train_acc'].append(train_acc)
        ft_history['val_loss'].append(avg_val_loss)
        ft_history['val_acc'].append(val_acc)
        ft_history['val_auc'].append(val_auc)
        ft_history['val_sens'].append(val_sens)
        ft_history['val_spec'].append(val_spec)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " ★"
        else:
            epochs_no_improve += 1
            marker = ""

        print(f"    FT Epoch [{epoch+1}/{FT_EPOCHS}] Train: {avg_train_loss:.4f}/{train_acc:.1f}% | "
              f"Val: {avg_val_loss:.4f}/{val_acc:.1f}% | AUC: {val_auc:.4f} | "
              f"Sens: {val_sens:.4f} | Spec: {val_spec:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"    Early stopping after {PATIENCE} epochs without AUC improvement.")
            break

    fold_histories.append(ft_history)

    # ── Load best model for this fold ──
    model.load_state_dict(best_model_state)
    print(f"\n  ✓ Fold {fold_idx+1} best val AUC: {best_val_auc:.4f}")

    # ── Optimal threshold from fold's validation set ──
    model.eval()
    val_y_true, val_y_probs = [], []
    with torch.no_grad():
        for imgs, labels in fold_val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
            else:
                logits = model(imgs)
            probs = torch.sigmoid(logits)
            val_y_true.extend(labels.long().cpu().numpy())
            val_y_probs.extend(probs.cpu().numpy())

    val_y_true = np.array(val_y_true)
    val_y_probs = np.array(val_y_probs)
    opt_thresh, opt_sens, opt_spec = find_optimal_threshold(
        val_y_true, val_y_probs, min_sensitivity=MIN_SENSITIVITY_TARGET
    )
    fold_thresholds.append(opt_thresh)
    print(f"  Optimal threshold (val): {opt_thresh:.4f}  (Sens={opt_sens:.4f}, Spec={opt_spec:.4f})")

    # ── Fold validation metrics ──
    val_auc_roc = roc_auc_score(val_y_true, val_y_probs)
    val_auc_pr = average_precision_score(val_y_true, val_y_probs)
    val_y_pred = (val_y_probs >= opt_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_y_true, val_y_pred).ravel()
    fold_val_results.append({
        'fold': fold_idx + 1,
        'auc_roc': val_auc_roc,
        'auc_pr': val_auc_pr,
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'accuracy': np.mean(val_y_true == val_y_pred),
        'threshold': opt_thresh,
    })

    # ── Evaluate on held-out test set ──
    test_y_true, test_y_probs = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).float()
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
            else:
                logits = model(imgs)
            probs = torch.sigmoid(logits)
            test_y_true.extend(labels.long().cpu().numpy())
            test_y_probs.extend(probs.cpu().numpy())

    test_y_true = np.array(test_y_true)
    test_y_probs = np.array(test_y_probs)
    fold_test_probs.append(test_y_probs)

    test_auc_roc = roc_auc_score(test_y_true, test_y_probs)
    test_auc_pr = average_precision_score(test_y_true, test_y_probs)
    test_y_pred = (test_y_probs >= opt_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_y_true, test_y_pred).ravel()
    test_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    test_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    test_acc = np.mean(test_y_true == test_y_pred)
    f1 = precision_recall_fscore_support(test_y_true, test_y_pred, average='binary', zero_division=0)[2]
    f2 = fbeta_score(test_y_true, test_y_pred, beta=2, average='binary', zero_division=0)

    fold_test_results.append({
        'fold': fold_idx + 1,
        'auc_roc': test_auc_roc,
        'auc_pr': test_auc_pr,
        'sensitivity': test_sens,
        'specificity': test_spec,
        'accuracy': test_acc,
        'f1': f1,
        'f2': f2,
        'threshold': opt_thresh,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    })

    print(f"  Test — AUC: {test_auc_roc:.4f} | Sens: {test_sens:.4f} | Spec: {test_spec:.4f} | Acc: {test_acc:.4f}")

    # ── Save best fold model for attention visualization ──
    if fold_idx == 0 or test_auc_roc > max([r['auc_roc'] for r in fold_test_results[:-1]], default=0):
        best_fold_model_path = "temp_best_vit_fold_model.pth"
        torch.save({
            'model_state_dict': best_model_state,
            'fold_idx': fold_idx + 1,
            'test_auc': test_auc_roc,
            'threshold': opt_thresh,
        }, best_fold_model_path)
        print(f"  ★ Saved as best fold model so far (test AUC: {test_auc_roc:.4f})")

    # ── Cleanup GPU memory ──
    del model, vit_copy, optimizer_ft, optimizer_warmup, scheduler_ft, scheduler_warmup
    del criterion, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print(f"  ALL {K_FOLDS} FOLDS COMPLETE")
print(f"{'='*70}")

## 7. Cross-Validation Results Summary

In [ ]:
df_val = pd.DataFrame(fold_val_results)
df_test = pd.DataFrame(fold_test_results)

print("=" * 80)
print("  STRATIFIED 5-FOLD CROSS-VALIDATION — RESULTS SUMMARY")
print("=" * 80)

# ── Per-fold results table ──
print("\n┌─── Per-Fold Test Set Results ───┐\n")
print(df_test[['fold', 'auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'f1', 'f2', 'threshold']].to_string(index=False, float_format='{:.4f}'.format))

# ── Aggregated metrics ──
metrics_to_report = ['auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'f1', 'f2']
print(f"\n┌─── Aggregated Test Metrics (mean ± std) ───┐\n")
for metric in metrics_to_report:
    vals = df_test[metric].values
    print(f"  {metric:<16s}: {vals.mean():.4f} ± {vals.std():.4f}  (range: {vals.min():.4f} – {vals.max():.4f})")

print(f"\n  Threshold (mean): {df_test['threshold'].mean():.4f} ± {df_test['threshold'].std():.4f}")

# ── Aggregated confusion matrix (sum across folds) ──
total_tp = df_test['tp'].sum()
total_tn = df_test['tn'].sum()
total_fp = df_test['fp'].sum()
total_fn = df_test['fn'].sum()
print(f"\n┌─── Aggregated Confusion Matrix (summed over {K_FOLDS} folds) ───┐")
print(f"  TN={total_tn:<5d} FP={total_fp:<5d}")
print(f"  FN={total_fn:<5d} TP={total_tp:<5d}")

# ── Validation set summary (sanity check) ──
print(f"\n┌─── Per-Fold Validation Set Results ───┐\n")
print(df_val[['fold', 'auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'threshold']].to_string(index=False, float_format='{:.4f}'.format))

print(f"\n  Val AUC-ROC (mean ± std): {df_val['auc_roc'].mean():.4f} ± {df_val['auc_roc'].std():.4f}")
print(f"  Val Sensitivity (mean ± std): {df_val['sensitivity'].mean():.4f} ± {df_val['sensitivity'].std():.4f}")
print("=" * 80)

## 8. Cross-Validation Visualizations

In [ ]:
# ── 8a. Per-fold metrics bar chart ──
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
metrics_plot = ['auc_roc', 'auc_pr', 'sensitivity', 'specificity', 'accuracy', 'f2']
titles = ['AUC-ROC', 'AUC-PR', 'Sensitivity', 'Specificity', 'Accuracy', 'F2 Score']
colors = ['darkorange', 'green', 'crimson', 'steelblue', 'mediumpurple', 'teal']

for ax, metric, title, color in zip(axes.flat, metrics_plot, titles, colors):
    vals = df_test[metric].values
    folds = df_test['fold'].values
    ax.bar(folds, vals, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.axhline(y=vals.mean(), color='black', linestyle='--', linewidth=2,
               label=f'Mean: {vals.mean():.4f}')
    ax.fill_between([0.5, K_FOLDS + 0.5],
                    vals.mean() - vals.std(), vals.mean() + vals.std(),
                    alpha=0.15, color='gray', label=f'±1 Std: {vals.std():.4f}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(title)
    ax.set_title(f'{title}\n{vals.mean():.4f} ± {vals.std():.4f}', fontweight='bold')
    ax.set_xticks(folds)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')
    # Set y-axis to focus on the data range
    y_min = max(0, vals.min() - 0.1)
    y_max = min(1.0, vals.max() + 0.1)
    ax.set_ylim(y_min, y_max)

plt.suptitle(f'Stratified {K_FOLDS}-Fold CV — Per-Fold Test Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8b. Training curves per fold (overlaid) ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, hist in enumerate(fold_histories):
    x = range(1, len(hist['train_loss']) + 1)
    axes[0, 0].plot(x, hist['val_loss'], marker='.', alpha=0.7, label=f'Fold {i+1}')
    axes[0, 1].plot(x, hist['val_acc'], marker='.', alpha=0.7, label=f'Fold {i+1}')
    axes[1, 0].plot(x, hist['val_auc'], marker='.', alpha=0.7, label=f'Fold {i+1}')
    axes[1, 1].plot(x, hist['val_sens'], marker='.', alpha=0.7, label=f'Fold {i+1}')

axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Val Loss')
axes[0, 0].set_title('Validation Loss per Fold'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Val Acc (%)')
axes[0, 1].set_title('Validation Accuracy per Fold'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Val AUC-ROC')
axes[1, 0].set_title('Validation AUC-ROC per Fold'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Val Sensitivity')
axes[1, 1].set_title('Validation Sensitivity per Fold'); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)
axes[1, 1].axhline(y=MIN_SENSITIVITY_TARGET, color='red', linestyle='--', alpha=0.5, label=f'Target: {MIN_SENSITIVITY_TARGET}')
axes[1, 1].legend()

plt.suptitle(f'Stratified {K_FOLDS}-Fold CV — Training Curves (Phase 2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Ensemble Evaluation (Averaged Fold Probabilities)

Average the test-set probabilities from all 5 fold models for a simple ensemble prediction.

In [ ]:
# Average probabilities across all fold models
ensemble_probs = np.mean(np.stack(fold_test_probs, axis=0), axis=0)

# Find optimal threshold on ensemble
ens_thresh, _, _ = find_optimal_threshold(
    test_y_true, ensemble_probs, min_sensitivity=MIN_SENSITIVITY_TARGET
)
ens_pred = (ensemble_probs >= ens_thresh).astype(int)

# Metrics
ens_auc = roc_auc_score(test_y_true, ensemble_probs)
ens_ap = average_precision_score(test_y_true, ensemble_probs)
ens_acc = np.mean(test_y_true == ens_pred)
tn, fp, fn, tp = confusion_matrix(test_y_true, ens_pred).ravel()
ens_f1 = precision_recall_fscore_support(test_y_true, ens_pred, average='binary', zero_division=0)[2]
ens_f2 = fbeta_score(test_y_true, ens_pred, beta=2, average='binary', zero_division=0)
ens_sens = tp / (tp + fn) if (tp + fn) > 0 else 0
ens_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
ens_ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
ens_npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print("=" * 70)
print("  ENSEMBLE (Mean of 5-Fold Probabilities) — Test Set")
print("=" * 70)
print(f"  Threshold: {ens_thresh:.4f}")
print(f"\n  SENSITIVITY METRICS (Malignant Detection):")
print(f"    Sensitivity (Recall):  {ens_sens:.4f} {'✓' if ens_sens >= MIN_SENSITIVITY_TARGET else '✗'}")
print(f"    PPV (Precision):       {ens_ppv:.4f}")
print(f"    F2 Score:              {ens_f2:.4f}")
print(f"\n  OVERALL METRICS:")
print(f"    Accuracy:              {ens_acc:.4f}")
print(f"    Specificity:           {ens_spec:.4f}")
print(f"    NPV:                   {ens_npv:.4f}")
print(f"    F1 Score:              {ens_f1:.4f}")
print(f"    AUC-ROC:               {ens_auc:.4f}")
print(f"    AUC-PR:                {ens_ap:.4f}")
print(f"\n  Confusion Matrix:")
print(f"    TN={tn:<4} FP={fp:<4}")
print(f"    FN={fn:<4} TP={tp:<4}")
print(f"\n{classification_report(test_y_true, ens_pred, target_names=CLASS_NAMES)}")
print("=" * 70)

# ── Plots ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm = confusion_matrix(test_y_true, ens_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title(f"Ensemble Confusion Matrix\nThreshold={ens_thresh:.3f}")

fpr, tpr, _ = roc_curve(test_y_true, ensemble_probs)
axes[1].plot(fpr, tpr, lw=2, color='darkorange', label=f'Ensemble ROC (AUC={ens_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
fpr_op = fp / (fp + tn) if (fp + tn) > 0 else 0
tpr_op = tp / (tp + fn) if (tp + fn) > 0 else 0
axes[1].plot(fpr_op, tpr_op, 'ro', markersize=10, label=f'Operating Point (Sens={tpr_op:.3f})')
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve (Ensemble)"); axes[1].legend(); axes[1].grid(alpha=0.3)

prec_c, rec_c, _ = precision_recall_curve(test_y_true, ensemble_probs)
axes[2].plot(rec_c, prec_c, lw=2, color='green', label=f'Ensemble PR (AP={ens_ap:.4f})')
axes[2].plot(tpr_op, ens_ppv, 'ro', markersize=10, label=f'Op. Point (Rec={tpr_op:.3f}, Prec={ens_ppv:.3f})')
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title("PR Curve (Ensemble)"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("5-Fold Ensemble — Test Set Evaluation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Attention Map Visualization (Best Fold Model)

Visualize what the model focuses on using ViT attention maps. We use the **best fold's model** (highest test AUC) and the **ensemble predictions** to identify TP/TN/FP/FN cases.

ViT attention extraction:
- Use `model.vit(pixel_values, output_attentions=True)`
- Extract attention from last layer: `attentions[-1]` shape [batch, heads, seq, seq]
- Average across heads, take CLS token row (index 0), reshape to 14x14 grid

In [ ]:
def get_vit_attention_map(vit_backbone, image_tensor):
    """
    Extract attention map from ViT model.
    
    Uses the attention weights from the last transformer layer,
    averaging across all heads and taking the CLS token's attention
    to all patch tokens.
    """
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    
    try:
        with torch.no_grad():
            outputs = vit_backbone(pixel_values=image_tensor, output_attentions=True)
        
        # Get attention from last layer: shape [batch, heads, seq_len, seq_len]
        # seq_len = 1 (CLS) + 196 (14x14 patches) = 197
        last_attn = outputs.attentions[-1]
        
        # Average across heads: [batch, seq_len, seq_len]
        attn_avg = last_attn.mean(dim=1)
        
        # Take CLS token row (index 0), excluding CLS token itself
        # Shape: [batch, seq_len-1] = [batch, 196]
        cls_attn = attn_avg[0, 0, 1:]  # Skip CLS token attention to itself
        
        # Reshape to 14x14 grid
        attn_map = cls_attn.reshape(14, 14).cpu().numpy()
        
        # Normalize to [0, 1]
        attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-6)
        
        return attn_map.astype(np.float32)
    except Exception as e:
        print(f"Error extracting attention: {e}")
        return None


def overlay_attention_on_image(image_np, attention_map, alpha=0.5):
    """Overlay attention heatmap on image."""
    h, w = image_np.shape[:2]
    attn_upsampled = cv2.resize(attention_map, (w, h))
    cmap = plt.get_cmap('jet')
    heatmap_rgb = cmap(attn_upsampled)[:, :, :3]
    heatmap_rgb = (heatmap_rgb * 255).astype(np.uint8)
    image_float = image_np.astype(np.float32) / 255.0
    heatmap_float = heatmap_rgb.astype(np.float32) / 255.0
    blended = (alpha * heatmap_float + (1 - alpha) * image_float)
    blended = np.clip(blended * 255, 0, 255).astype(np.uint8)
    return blended


def visualize_attention_quadrant(quadrant_name, quadrant_indices, vit_backbone,
                                 class_name_true, class_name_pred, threshold=0.5, n_samples=3):
    """Visualize ViT attention for classification results."""
    if len(quadrant_indices) == 0:
        print(f"\n  No {quadrant_name} samples.")
        return
    sample_indices = np.random.choice(quadrant_indices, size=min(n_samples, len(quadrant_indices)), replace=False)
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, 3)
    fig.suptitle(f"{quadrant_name}\nTrue: {class_name_true} | Pred: {class_name_pred}",
                 fontsize=14, fontweight='bold')
    for row, test_idx in enumerate(sample_indices):
        row_data = test_manifest.iloc[test_idx]
        cancer_type = row_data['cancer_type']
        prob = ensemble_probs[test_idx]
        filenames = str(row_data['dscope_files']).split(';')
        dscope_path = os.path.join(IMAGES_DIR, filenames[0]) if filenames else None
        try:
            if not dscope_path or not os.path.exists(dscope_path):
                raise FileNotFoundError(f"Dermoscope image not found")
            dscope_pil = Image.open(dscope_path).convert("RGB")
            dscope_orig = np.array(dscope_pil)
            dscope_tensor = eval_transform(dscope_pil)
            attn_dscope = get_vit_attention_map(vit_backbone, dscope_tensor)
            if attn_dscope is not None:
                dscope_with_attn = overlay_attention_on_image(dscope_orig, attn_dscope, alpha=0.4)
            else:
                dscope_with_attn = dscope_orig
            axes[row, 0].imshow(dscope_orig)
            axes[row, 0].axis('off')
            axes[row, 0].set_title(f"Dermoscope Original\n{cancer_type}", fontsize=10)
            axes[row, 1].imshow(dscope_with_attn)
            axes[row, 1].axis('off')
            axes[row, 1].set_title(f"ViT Attention Map\n(Red/Yellow = High Attention)",
                                  fontsize=10, fontweight='bold', color='darkblue')
            ax_bar = axes[row, 2]
            color = 'crimson' if prob >= threshold else 'steelblue'
            pred_label = 'Malignant' if prob >= threshold else 'Benign'
            ax_bar.barh([0], [prob], color=color, alpha=0.8, edgecolor='black', linewidth=2)
            ax_bar.axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.3f}')
            ax_bar.set_xlim(0, 1)
            ax_bar.set_yticks([])
            ax_bar.set_xlabel('Probability (Ensemble)', fontsize=10, fontweight='bold')
            ax_bar.set_title(f"Model Confidence\nProb: {prob:.3f}\nPred: {pred_label}",
                            fontsize=10, fontweight='bold')
            ax_bar.legend(loc='upper right', fontsize=8)
            ax_bar.grid(axis='x', alpha=0.3)
            ax_bar.text(prob, 0, f'{prob:.3f}', ha='center', va='center',
                       fontsize=10, fontweight='bold', color='white')
        except Exception as e:
            error_msg = str(e)[:40]
            for j in range(3):
                axes[row, j].text(0.5, 0.5, f"Error:\n{error_msg}", ha='center', va='center', fontsize=9)
                axes[row, j].axis('off')
    plt.tight_layout()
    plt.show()


print("✓ ViT attention visualization utilities defined.")

In [ ]:
# Load the best fold model for attention visualization
checkpoint = torch.load(best_fold_model_path, map_location=DEVICE)
best_fold_idx = checkpoint['fold_idx']
best_fold_auc = checkpoint['test_auc']
best_fold_threshold = checkpoint['threshold']

print(f"Loading best fold model for attention visualization:")
print(f"  Fold: {best_fold_idx} / {K_FOLDS}")
print(f"  Test AUC: {best_fold_auc:.4f}")
print(f"  Threshold: {best_fold_threshold:.4f}")

# Reconstruct model
viz_vit = copy.deepcopy(vit_model_ref)
viz_model = BinaryViTModel(
    vit_model=viz_vit, embed_dim=embed_dim, hidden=512, dropout=0.3
).to(DEVICE)
viz_model.load_state_dict(checkpoint['model_state_dict'])
viz_model.eval()

print(f"✓ Model loaded and ready for visualization")

In [ ]:
# Identify TP/TN/FP/FN based on ensemble predictions
ensemble_y_pred = (ensemble_probs >= ens_thresh).astype(int)

tp_idx = np.where((test_y_true == 1) & (ensemble_y_pred == 1))[0]
tn_idx = np.where((test_y_true == 0) & (ensemble_y_pred == 0))[0]
fp_idx = np.where((test_y_true == 0) & (ensemble_y_pred == 1))[0]
fn_idx = np.where((test_y_true == 1) & (ensemble_y_pred == 0))[0]

print("\n" + "="*80)
print(f"  K-FOLD CV — ATTENTION MAP ANALYSIS (Fold {best_fold_idx} Model)")
print("="*80)
print(f"\nUsing ensemble threshold = {ens_thresh:.3f} (for ≥{MIN_SENSITIVITY_TARGET*100:.0f}% sensitivity)")
print(f"\nConfusion matrix breakdown:")
print(f"  TP: {len(tp_idx)} | TN: {len(tn_idx)} | FP: {len(fp_idx)} | FN: {len(fn_idx)}")
print(f"\nNote: Using ensemble predictions but visualizing attention from Fold {best_fold_idx} model")
print(f"      (best test AUC = {best_fold_auc:.4f})")

print("\n" + "-"*80)
print("TRUE POSITIVES (Correctly Detected Malignant)")
print("-"*80)
visualize_attention_quadrant("TRUE POSITIVES", tp_idx, viz_model.vit,
                             "Malignant", "Malignant ✓", threshold=ens_thresh, n_samples=3)

print("\n" + "-"*80)
print("TRUE NEGATIVES (Correctly Detected Benign)")
print("-"*80)
visualize_attention_quadrant("TRUE NEGATIVES", tn_idx, viz_model.vit,
                             "Benign", "Benign ✓", threshold=ens_thresh, n_samples=3)

print("\n" + "-"*80)
print("FALSE POSITIVES (Benign Misclassified as Malignant)")
print("-"*80)
visualize_attention_quadrant("FALSE POSITIVES", fp_idx, viz_model.vit,
                             "Benign", "Malignant (WRONG)", threshold=ens_thresh, n_samples=3)

print("\n" + "-"*80)
print("FALSE NEGATIVES (Malignant Missed - CRITICAL)")
print("-"*80)
visualize_attention_quadrant("FALSE NEGATIVES", fn_idx, viz_model.vit,
                             "Malignant", "Benign (WRONG)", threshold=ens_thresh, n_samples=3)

## 11. Cleanup

In [ ]:
# Remove temporary model file
import os
if os.path.exists(best_fold_model_path):
    os.remove(best_fold_model_path)
    print(f"✓ Removed temporary model file: {best_fold_model_path}")

print("\n✓ ViT-Small Full Fine-Tune experiment complete!")